# Analyze Study (Binary Classification)

Comprehensive post-hoc analysis of a completed Octopus binary classification study.

- **Study Details** -- load, validate, and summarize the study configuration and workflow structure.
- **Target Metric Performance** -- development-set performance across outer splits for all prediction tasks and result types (e.g. best, ensemble selection). Feature selection tasks are skipped automatically.
- **Selected Features** -- number of selected features per task and outer split, plus feature selection frequency across splits.
- **Test Performance** -- for a selected task: test-set metrics, AUCROC curves, confusion matrices, and feature importances (permutation and SHAP).

## Imports

In [ ]:
from octopus.poststudy import OctoTestEvaluator, load_study_information
from octopus.poststudy.notebook import (
    display_table,
    show_aucroc,
    show_confusionmatrix,
    show_fi,
    show_performance_tables,
    show_selected_features_tables,
    show_study_overview,
    show_testset_performance,
)
from octopus.poststudy.plots import (
    feature_count_plot,
    feature_frequency_plot,
    performance_plot,
)
from octopus.poststudy.study_io import find_latest_study
from octopus.poststudy.tables import (
    get_performance,
    get_selected_features,
    workflow_graph,
)

## Input

For the analysis, a path to a saved study directory needs to be provided. Octopus saves studies in directories named `<prefix>-YYYYMMDD_HHMMSS`. The helper `find_latest_study` scans the studies root folder and returns the most recent directory matching the given name prefix. Overwrite study_directory if you want to use a specific study path.

In [ ]:
study_name_prefix = "wf_octo_mrmr_octo"
study_directory = find_latest_study("../studies", study_name_prefix)
print(f"Using study: {study_directory}")

## Study Details

`load_study_information` validates the study directory (checks outersplit and task directories) and returns a `StudyInfo` object used by all downstream analysis functions. `show_study_overview` prints a concise summary. `workflow_graph` renders the task dependency tree.

In [ ]:
study_info = load_study_information(study_directory)

In [ ]:
show_study_overview(study_info)

In [ ]:
print(workflow_graph(study_info))

## Performance for all Tasks

Development-set performance per outer split for all prediction tasks (octo, autogluon). Feature selection tasks (mrmr, roc, boruta) are skipped automatically. If a task has multiple result types (e.g. `best` and `ensemble_selection`), each is shown as a separate table. 

In [ ]:
perf = get_performance(study_info)

In [ ]:
fig = performance_plot(perf)
fig.show()

In [ ]:
show_performance_tables(perf)

## Selected Features

Number of selected features per outer split and task, plus how frequently each feature was selected across splits. Covers all tasks and result types.

In [ ]:
feature_table, frequency_table = get_selected_features(study_info)

In [ ]:
fig = feature_count_plot(feature_table)
fig.show()

In [ ]:
fig = feature_frequency_plot(frequency_table)
fig.show()

In [ ]:
#show_selected_features_tables(feature_table, frequency_table)

## Model Performance on Test Dataset for a given Task

The sections above use the **dev** partition -- these results should guide model selection, hyperparameter tuning, and feature selection decisions.

The sections below evaluate a **single selected task** on the held-out **test** partition. Only look at test results after all modelling decisions have been made -- if test results lead you to change the model, the test set is no longer independent and reported scores lose their validity.

`OctoTestEvaluator` loads the fitted models and the stored train/test splits for the selected task. All subsequent calls (performance, AUCROC, confusion matrix, feature importance) use this predictor.

In [ ]:
# Input: selected metrics for performance overview
metrics = ["AUCROC", "ACCBAL", "ACC", "F1", "AUCPR", "NEGBRIERSCORE"]
print("Selected metrics: ", metrics)

### Test performance for given task and selected metrics

Test-set performance per outer split (each model scored only on its own held-out test fold) with a mean row.

In [ ]:
# load predictor object (OctoTestEvaluator includes stored test data)
task_predictor_test = OctoTestEvaluator(
    study_info=study_info, task_id=0, result_type="best"
)
testset_performance = show_testset_performance(
    predictor=task_predictor_test, metrics=metrics
)

### AUCROC Plots

Receiver Operating Characteristic curves on the held-out test data. Shows a merged curve (all predictions pooled) and an averaged curve (mean +/- 1 std across outer splits). Set `show_individual=True` to also display per-split curves.

In [ ]:
show_aucroc(task_predictor_test, show_individual=True)

### Confusion Matrix

Absolute and relative confusion matrices per outer split, plus per-split and overall mean performance metrics. The classification threshold defaults to 0.5.

In [ ]:
show_confusionmatrix(task_predictor_test, threshold=0.5, metrics=metrics)

### Test Feature Importances

Feature importance computed on the held-out test data using the final fitted models. Two methods are available: permutation-based importance and SHAP values. Both produce per-split and ensemble (averaged across splits) results.

#### Calculate Permutation Feature Importances

In [ ]:
# Permutation feature importances on test data using final models
# calculate_fi() returns the full FI table (per-split + ensemble rows)
print("PFI calculation running.....")
fi_table_perm = task_predictor_test.calculate_fi(
    fi_type="group_permutation", n_repeats=3
)

#### Ensemble Feature Importances (table + plot)

In [ ]:
# show_fi combines show_overall_fi_table + show_overall_fi_plot into one call
fi_ensemble_perm = show_fi(fi_table_perm)
fi_ensemble_perm.head(10)

#### Individual Per-Split Feature Importances

In [ ]:
# Access per-split feature importances via the fi_source column
for split_id in task_predictor_test.study_info.outersplits:
    split_fi = fi_table_perm[fi_table_perm["fi_source"] == split_id].copy()
    print(f"\n=== Outersplit {split_id} ===")
    display_table(
        split_fi[["feature", "importance_mean", "importance_std", "p_value"]].head(10)
    )

#### Calculate SHAP Feature Importances

Available `shap_type` options:

- `'kernel'` — Model-agnostic (KernelExplainer). Works with any model. Default.
- `'permutation'` — Model-agnostic (PermutationExplainer). Permutation-based approach.
- `'exact'` — Model-agnostic (ExactExplainer). Exact SHAP values (slowest, most accurate).

In [ ]:
# SHAP feature importances on test data using final models
# calculate_fi() returns the full FI table (per-split + ensemble rows)
# Available shap_type: 'kernel' (default), 'permutation', 'exact'
fi_table_shap = task_predictor_test.calculate_fi(
    fi_type="shap", shap_type="kernel"
)

In [ ]:
fi_ensemble_shap = show_fi(fi_table_shap)
fi_ensemble_shap.head(10)